# 📡 TradePulse — NLP Demo
**Economic Power Shift Detector | Phase 1 & 2 Demo**

This notebook demonstrates the core NLP pipeline:
1. Fetch live bilateral news from RSS feeds
2. Run FinBERT bilateral sentiment classification
3. Compute Leverage Signal score

Run each cell in order. GPU is enabled for faster inference.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install transformers torch feedparser streamlit pyngrok -q
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Clone / upload project files ─────────────────────────────────────
# Option A: If you uploaded the zip
# !unzip tradepulse.zip -d tradepulse

# Option B: Create inline (paste your core files here)
import os
os.makedirs('tradepulse/core', exist_ok=True)
os.makedirs('tradepulse/demo', exist_ok=True)
print('📁 Directories ready')

In [ ]:
# ── Cell 3: Test FinBERT directly (no app needed) ────────────────────────────
from transformers import pipeline

print('Loading FinBERT...')
clf = pipeline('text-classification', model='ProsusAI/finbert',
               return_all_scores=True, truncation=True, max_length=512)
print('✅ Model loaded')

LABEL_MAP = {'positive': 'cooperative', 'negative': 'adversarial', 'neutral': 'neutral'}

test_headlines = [
    'Regarding the economic relationship between India and China: '
    'India imposes anti-dumping duty on Chinese steel imports, escalating trade tensions',

    'Regarding the economic relationship between India and China: '
    'India and China agree to resume trade normalisation talks in Beijing',

    'Regarding the economic relationship between India and China: '
    "India's trade deficit with China widens to record $85 billion this fiscal year",

    'Regarding the economic relationship between India and China: '
    'India bans 47 Chinese mobile applications citing national security concerns',

    'Regarding the economic relationship between India and China: '
    'India and China sign new bilateral investment facilitation agreement',
]

print('\n' + '='*65)
print('TradePulse — Bilateral Sentiment Analysis: India ↔ China')
print('='*65)

numerics = []
for text in test_headlines:
    result = clf(text)[0]
    best = max(result, key=lambda x: x['score'])
    label = LABEL_MAP[best['label'].lower()]
    score_num = {'cooperative': 1.0, 'neutral': 0.0, 'adversarial': -1.0}[label]
    numerics.append(score_num)
    emoji = {'cooperative': '🟢', 'neutral': '🟡', 'adversarial': '🔴'}[label]
    # Print original headline (without context prefix)
    headline = text.split(': ', 1)[1]
    print(f'\n{emoji} [{label.upper():>13}] conf={best["score"]:.3f}')
    print(f'   {headline[:70]}')

mean_score = sum(numerics) / len(numerics)
print('\n' + '='*65)
print(f'MEAN SENTIMENT SCORE: {mean_score:+.3f}')
overall = 'ADVERSARIAL' if mean_score < -0.2 else ('COOPERATIVE' if mean_score > 0.2 else 'NEUTRAL')
print(f'OVERALL LABEL:        {overall}')
print('='*65)

In [ ]:
# ── Cell 4: Leverage Score computation ───────────────────────────────────────
# Trade asymmetry (World Bank 2022-23 data)
TRADE_DATA = {
    ('IN','CN'): (0.034, 0.032),  # India→China, China→India export shares
    ('IN','US'): (0.178, 0.021),
    ('IN','RU'): (0.021, 0.189),
    ('IN','EU'): (0.121, 0.018),
}

def compute_leverage(country_a, country_b, sentiment_mean):
    key = (country_a, country_b) if country_a < country_b else (country_b, country_a)
    shares = TRADE_DATA.get(key, (0.05, 0.05))
    share_a, share_b = shares
    if country_a > country_b:  # reversed key
        share_a, share_b = share_b, share_a

    asymmetry = abs(share_a - share_b)
    trade_score = min(asymmetry / 0.20, 1.0)

    # Sentiment: adversarial (-1) → high score (1.0), cooperative (+1) → low (0.0)
    sentiment_score = (1.0 - (sentiment_mean + 1.0) / 2.0)

    stance_score = 0.5  # neutral default (Phase 2 will compute this)

    leverage_raw = 0.40 * trade_score + 0.35 * sentiment_score + 0.25 * stance_score
    leverage_10  = round(leverage_raw * 10, 2)

    label = ('Stable' if leverage_10 < 2.5 else
             'Watchlist' if leverage_10 < 5.0 else
             'Elevated' if leverage_10 < 7.5 else 'Critical')

    return leverage_10, label, trade_score, sentiment_score

# Test on India ↔ China with our computed sentiment
lev, label, ts, ss = compute_leverage('IN', 'CN', mean_score)
print('\n' + '='*55)
print(f'  LEVERAGE SIGNAL: India ↔ China')
print('='*55)
print(f'  Score          :  {lev} / 10  [{label}]')
print(f'  Trade Asymmetry:  {ts:.3f}  (weight 40%)')
print(f'  Sentiment NLP  :  {ss:.3f}  (weight 35%)')
print(f'  Policy Stance  :  0.500  (weight 25%) — Phase 2 pending')
print('='*55)
print(f'\n  Interpretation: The India-China bilateral leverage')
print(f'  relationship shows an {label.lower()} imbalance.')
print(f'  Sentiment driven by adversarial trade news;')
print(f'  India holds a {abs(0.034-0.032):.1%} structural export asymmetry.')

In [ ]:
# ── Cell 5: Live RSS news fetch ───────────────────────────────────────────────
!pip install feedparser -q
import feedparser, re

ALIASES = {
    'IN': ['india','indian','new delhi','modi','rbi'],
    'CN': ['china','chinese','beijing','xi jinping','pboc'],
    'US': ['united states','america','american','washington','ustr'],
    'RU': ['russia','russian','moscow','putin'],
}

FEEDS = [
    ('Reuters Business', 'https://feeds.reuters.com/reuters/businessNews'),
    ('PIB India',        'https://pib.gov.in/RssMain.aspx?ModId=6&Lang=1&Regid=3'),
    ('BBC Business',     'http://feeds.bbci.co.uk/news/business/rss.xml'),
]

def detect(text, iso):
    return any(a in text.lower() for a in ALIASES[iso])

print('Fetching live news...')
articles = []
for name, url in FEEDS:
    try:
        feed = feedparser.parse(url)
        for e in feed.entries[:15]:
            title = e.get('title','')
            body  = re.sub(r'<[^>]+>','',e.get('summary',''))
            articles.append({'source': name, 'title': title, 'text': f'{title}. {body}'})
    except: pass

relevant = [a for a in articles if detect(a['text'],'IN') and detect(a['text'],'CN')]
print(f'Total fetched: {len(articles)} | India↔China relevant: {len(relevant)}')

if relevant:
    print('\nLive headlines:')
    for a in relevant[:5]:
        print(f'  [{a["source"]}] {a["title"][:75]}')
else:
    print('\nNo live bilateral articles found today — demo corpus used in Cell 3')

In [ ]:
# ── Cell 6: Run Streamlit app via ngrok (optional — for live demo) ────────────
# Uncomment and run this cell to get a public URL for your app
# Requirements: sign up at https://ngrok.com, get your authtoken

# NGROK_TOKEN = 'your_ngrok_authtoken_here'
# !ngrok authtoken {NGROK_TOKEN}

# import subprocess, threading, time
# from pyngrok import ngrok

# subprocess.Popen(['streamlit', 'run', 'tradepulse/demo/demo_app.py',
#                   '--server.port', '8501', '--server.headless', 'true'])
# time.sleep(3)
# tunnel = ngrok.connect(8501)
# print(f'\n🌐 TradePulse LIVE at: {tunnel.public_url}')

print('Uncomment the lines above and add your ngrok token to get a live public URL.')
print('Without ngrok, run locally with: streamlit run demo/demo_app.py')